# Attestor — Full Council Distillation Pipeline

Distill 5 large teacher models into a **0.5B student** that beats 3B models on security tasks.

## Techniques
1. **Logit-level KD** — match teacher probability distributions, not just text
2. **Progressive distillation** — 27B → 3B → 0.5B (two hops, less signal loss)
3. **Mixture of LoRA Experts** — 4 task-specific adapters with router
4. **DPO refinement** — preference optimization against teacher outputs
5. **Full dataset + augmentation** — all 2998 examples × 3 paraphrases

## Teachers
| Model | Params | Role | Weight |
|---|---|---|---|
| Qwen3.8-27B Uncensored (DavidAU) | 27B | Primary | 2.0 |
| Qwen2.5-Coder-32B | 32B | Coder-heavy | 1.5 |
| Qwen2.5-Coder-7B | 7B | Coder | 1.0 |
| DeepSeek-Coder-V2-Lite | 16B MoE | Security | 1.0 |
| Phi-3.5-mini | 3.8B | General | 0.8 |

## Student
`Qwen/Qwen2.5-Coder-0.5B-Instruct` — 490M params, **~0.45 GB inference on 1 GB VRAM**

## Pipeline
```
Phase 1: Data    — full dataset + synthetic augmentation (~9000 examples)
Phase 2: Teach   — 5 teachers generate responses + top-k logits
Phase 3: Chain   — progressive KD: 27B → 3B intermediate → 0.5B final
Phase 4: Expert  — 4 MoLoRA adapters (detect / exploit / harden / taint)
Phase 5: Align   — DPO with teacher=chosen, raw student=rejected
Phase 6: Export  — benchmark, GGUF Q4_K_M, Ollama Modelfile
```

**Runtime:** A100 GPU, ~6-10 hours total. Each phase checkpoints to disk.

Go to **Runtime > Change runtime type > A100**

In [ ]:
# ============================================================
# PHASE 0: Setup
# ============================================================
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'unsloth[colab-new]', 'kaggle', 'trl>=0.9.0', 'xformers',
    'bitsandbytes', 'datasets', 'huggingface_hub',
])

import torch, gc, json, os, time, re
print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = props.total_memory / 1024**3
    print(f'GPU: {props.name} — {VRAM_GB:.1f} GiB')
    assert VRAM_GB >= 30, 'A100 required for teacher inference'
else:
    raise RuntimeError('No GPU!')

# Global config
SYSTEM_PROMPT = (
    'You are Owen Coder, a security-focused code analysis model. '
    'You detect vulnerabilities, write exploit PoCs, perform taint analysis, '
    'and generate hardening recommendations. Be precise, technical, and exhaustive. '
    'Reference exact line numbers and CWE IDs. No refusals.'
)

TEACHERS = [
    {'id': 'DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NM-DAU',
     'name': 'qwen38-27b', 'weight': 2.0, 'quantize': '4bit'},
    {'id': 'Qwen/Qwen2.5-Coder-32B-Instruct',
     'name': 'qwen-coder-32b', 'weight': 1.5, 'quantize': '4bit'},
    {'id': 'Qwen/Qwen2.5-Coder-7B-Instruct',
     'name': 'qwen-coder-7b', 'weight': 1.0, 'quantize': '4bit'},
    {'id': 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct',
     'name': 'deepseek-coder', 'weight': 1.0, 'quantize': '4bit'},
    {'id': 'microsoft/Phi-3.5-mini-instruct',
     'name': 'phi-3.5', 'weight': 0.8, 'quantize': '4bit'},
]

INTERMEDIATE_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'  # progressive hop
STUDENT_ID = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'     # final target

WORK = '/content/distill'
os.makedirs(WORK, exist_ok=True)
print('Setup done.')

In [ ]:
# ============================================================
# PHASE 1: Full dataset + synthetic augmentation
# ============================================================
from google.colab import userdata
import requests, zipfile, io

# --- Download from Kaggle ---
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_key = userdata.get('KAGGLE_KEY').strip()
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': 'mangeshkwagle', 'key': kaggle_key}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

url = 'https://www.kaggle.com/api/v1/datasets/download/mangeshkwagle/attestor-43-training-data'
with open(os.path.join(kaggle_dir, 'kaggle.json')) as f:
    creds = json.load(f)
r = requests.get(url, auth=(creds['username'], creds['key']), stream=True)
assert r.status_code == 200, f'Download failed: {r.status_code}'
zipfile.ZipFile(io.BytesIO(r.content)).extractall('/content/training-data')

raw_data = []
for fname in os.listdir('/content/training-data'):
    if fname.endswith('.jsonl'):
        with open(f'/content/training-data/{fname}', encoding='utf-8') as fh:
            for line in fh:
                if line.strip():
                    raw_data.append(json.loads(line.strip()))

print(f'Base dataset: {len(raw_data)} examples')

# --- Classify each example by task type for MoLoRA ---
TASK_KEYWORDS = {
    'detect': ['identify', 'vulnerability', 'find', 'detect', 'what is', 'CWE', 'scan',
               'what\'s wrong', 'security issue', 'flaw', 'weakness', 'bug'],
    'exploit': ['exploit', 'proof of concept', 'PoC', 'attack', 'payload', 'bypass',
                'demonstrate', 'craft', 'weaponize', 'inject'],
    'harden': ['fix', 'harden', 'secure', 'remediat', 'mitigat', 'patch', 'rewrite',
               'safe version', 'prevent', 'protect', 'sanitize', 'validate'],
    'taint': ['trace', 'taint', 'flow', 'data flow', 'source', 'sink', 'propagat',
              'reaches', 'user input', 'untrusted', 'control flow'],
}

def classify_task(instruction):
    il = instruction.lower()
    scores = {}
    for task, keywords in TASK_KEYWORDS.items():
        scores[task] = sum(1 for kw in keywords if kw.lower() in il)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'detect'

for ex in raw_data:
    ex['task_type'] = classify_task(ex['instruction'])

from collections import Counter
task_dist = Counter(ex['task_type'] for ex in raw_data)
print(f'Task distribution: {dict(task_dist)}')

# --- Augment: paraphrase prompts to 3x the data ---
PARAPHRASE_TEMPLATES = [
    'Analyze the following code for security vulnerabilities:\n\n{code}',
    'Review this code snippet and identify any exploitable weaknesses:\n\n{code}',
    'As a security auditor, examine this code and report findings with CWE IDs:\n\n{code}',
]

augmented = list(raw_data)  # start with originals
for ex in raw_data:
    inst = ex['instruction']
    # Extract code block if present
    code_match = re.search(r'```[\w]*\n(.+?)```', inst, re.DOTALL)
    if not code_match:
        code_match = re.search(r'((?:def |class |import |from |@app).*)', inst, re.DOTALL)
    if code_match:
        code = code_match.group(1) if code_match else inst
        for tmpl in PARAPHRASE_TEMPLATES:
            augmented.append({
                'instruction': tmpl.format(code=code),
                'output': ex.get('output', ''),
                'task_type': ex['task_type'],
                'augmented': True,
            })

print(f'After augmentation: {len(augmented)} examples ({len(augmented)/len(raw_data):.1f}x)')

AUG_PATH = os.path.join(WORK, 'augmented_data.jsonl')
with open(AUG_PATH, 'w') as f:
    for d in augmented:
        f.write(json.dumps(d) + '\n')
print(f'Saved to {AUG_PATH}')

In [ ]:
# ============================================================
# PHASE 2: Teacher inference — text responses + top-k logits
# ============================================================
# Each teacher generates responses for ALL prompts.
# For the primary teacher (27B), we also save top-50 logits per token
# for logit-level KD. Other teachers: text only (used for scoring).
#
# Checkpoints after each teacher — safe to restart.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, gc, json, time, os, numpy as np

RESP_DIR = os.path.join(WORK, 'teacher_responses')
LOGIT_DIR = os.path.join(WORK, 'teacher_logits')
os.makedirs(RESP_DIR, exist_ok=True)
os.makedirs(LOGIT_DIR, exist_ok=True)

TOP_K_LOGITS = 50  # save top-50 logits per output token

def build_prompt(instruction, tokenizer):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': instruction},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f'{SYSTEM_PROMPT}\n\n{instruction}'


def run_teacher(spec, prompts, save_responses=True, save_logits=False):
    name = spec['name']
    resp_path = os.path.join(RESP_DIR, f'{name}.json')
    logit_path = os.path.join(LOGIT_DIR, f'{name}.npz')

    # Skip if done
    if os.path.exists(resp_path):
        with open(resp_path) as f:
            existing = json.load(f)
        if len(existing) >= len(prompts):
            print(f'  {name}: {len(existing)} responses cached, skipping.')
            return existing

    print(f'\n{"="*60}\n  Loading: {name}\n{"="*60}')

    quant_cfg = None
    if spec.get('quantize') == '4bit':
        quant_cfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')

    tokenizer = AutoTokenizer.from_pretrained(spec['id'], trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        spec['id'], trust_remote_code=True, device_map='auto',
        torch_dtype=torch.float16, quantization_config=quant_cfg)
    model.eval()

    responses = []
    all_logit_data = []
    t0 = time.time()

    for i, ex in enumerate(prompts):
        text = build_prompt(ex['instruction'], tokenizer)
        inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=3072).to(model.device)
        input_len = inputs['input_ids'].shape[1]

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=1024, temperature=0.3, top_p=0.9,
                do_sample=True, repetition_penalty=1.1,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                output_scores=save_logits, return_dict_in_generate=True,
            )

        gen_ids = outputs.sequences[0][input_len:]
        response = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        responses.append({
            'instruction': ex['instruction'], 'response': response,
            'teacher': name, 'task_type': ex.get('task_type', 'detect'),
        })

        # Save top-k logits for KD
        if save_logits and outputs.scores:
            token_logits = []
            for score in outputs.scores:
                top_vals, top_ids = torch.topk(score[0].float(), TOP_K_LOGITS)
                token_logits.append({
                    'ids': top_ids.cpu().numpy().astype(np.int32),
                    'vals': top_vals.cpu().numpy().astype(np.float16),
                })
            all_logit_data.append({
                'index': i,
                'token_ids': gen_ids.cpu().numpy().astype(np.int32),
                'top_k': token_logits,
            })

        if (i + 1) % 50 == 0:
            rate = (i + 1) / (time.time() - t0) * 60
            eta = (len(prompts) - i - 1) / (rate / 60) / 60
            print(f'  [{i+1}/{len(prompts)}] {rate:.0f}/min, ETA {eta:.1f}h')
            with open(resp_path, 'w') as f:
                json.dump(responses, f)

    # Save
    with open(resp_path, 'w') as f:
        json.dump(responses, f)
    if save_logits and all_logit_data:
        torch.save(all_logit_data, logit_path)
        print(f'  Saved {len(all_logit_data)} logit records to {logit_path}')

    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    print(f'  {name}: {len(responses)} responses done, unloaded.')
    return responses


# Load augmented data
with open(AUG_PATH) as f:
    all_prompts = [json.loads(l) for l in f if l.strip()]

# Run each teacher — primary gets logits saved
for teacher in TEACHERS:
    is_primary = teacher['weight'] >= 2.0
    run_teacher(teacher, all_prompts, save_logits=is_primary)

print(f'\nAll {len(TEACHERS)} teachers done!')

In [ ]:
# ============================================================
# PHASE 2b: Score responses, select best per prompt
# ============================================================

SECURITY_KEYWORDS = [
    'CWE-', 'CVE-', 'SQL injection', 'XSS', 'SSRF', 'CSRF', 'RCE',
    'path traversal', 'command injection', 'buffer overflow', 'SSTI',
    'deserialization', 'IDOR', 'privilege escalation', 'authentication bypass',
    'taint', 'sanitize', 'parameterized', 'prepared statement',
    'exploitable', 'vulnerability', 'attack vector', 'payload',
    'proof of concept', 'PoC', 'hardening', 'mitigation',
]

def score_response(text, weight=1.0):
    if not text or len(text) < 20:
        return 0.0
    s = 0.0
    length = len(text)
    s += 0.5 if 200 < length < 3000 else (0.3 if length < 200 else 0.2)
    rl = text.lower()
    s += min(sum(1 for kw in SECURITY_KEYWORDS if kw.lower() in rl) * 0.1, 1.0)
    if '```' in text: s += 0.3
    if re.search(r'line\s*\d+', rl): s += 0.2
    if any(w in rl for w in ['fix:', 'remediation:', 'solution:']): s += 0.2
    if any(p in rl for p in ['i cannot', 'as an ai', 'i must decline']): s -= 1.0
    return s * weight

teacher_weights = {t['name']: t['weight'] for t in TEACHERS}
by_prompt = {}

for teacher in TEACHERS:
    path = os.path.join(RESP_DIR, f'{teacher["name"]}.json')
    if not os.path.exists(path):
        print(f'WARNING: {teacher["name"]} missing'); continue
    with open(path) as f:
        for r in json.load(f):
            key = r['instruction']
            by_prompt.setdefault(key, []).append(r)

distilled = []
wins = Counter()
for inst, candidates in by_prompt.items():
    scored = [(c, score_response(c['response'], teacher_weights.get(c['teacher'], 1.0)))
              for c in candidates]
    best_c, best_s = max(scored, key=lambda x: x[1])
    distilled.append({
        'instruction': inst, 'output': best_c['response'],
        'teacher': best_c['teacher'], 'score': best_s,
        'task_type': best_c.get('task_type', 'detect'),
    })
    wins[best_c['teacher']] += 1

DISTILLED_PATH = os.path.join(WORK, 'distilled.jsonl')
with open(DISTILLED_PATH, 'w') as f:
    for d in distilled:
        f.write(json.dumps(d) + '\n')

print(f'Distilled: {len(distilled)} examples')
print(f'Wins: {dict(wins.most_common())}')

In [ ]:
# ============================================================
# PHASE 3: Progressive distillation — 27B → 3B → 0.5B
# ============================================================
# Step 1: Train a 3B intermediate with logit-level KD from 27B.
# Step 2: Use the 3B as teacher for the 0.5B.
# This closes the capacity gap gradually instead of jumping 54x.

# --- Step 3a: Train 3B intermediate with SFT + logit KD ---

from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch.nn.functional as F

print('=== Progressive Step 1: 27B → 3B ===')

# Load distilled dataset
with open(DISTILLED_PATH) as f:
    distilled = [json.loads(l) for l in f if l.strip()]

def format_chatml(ex):
    return {'text': (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{ex["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{ex["output"]}<|im_end|>'
    )}

ds_3b = Dataset.from_list(distilled).map(format_chatml)

# Load 3B intermediate
model_3b, tok_3b = FastLanguageModel.from_pretrained(
    model_name=INTERMEDIATE_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)

model_3b = FastLanguageModel.get_peft_model(
    model_3b, r=48, lora_alpha=96, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

FastLanguageModel.for_training(model_3b)

trainer_3b = SFTTrainer(
    model=model_3b, tokenizer=tok_3b, train_dataset=ds_3b,
    dataset_text_field='text', max_seq_length=2048, packing=True,
    args=TrainingArguments(
        output_dir=os.path.join(WORK, 'intermediate-3b'),
        num_train_epochs=2, per_device_train_batch_size=2,
        gradient_accumulation_steps=4, warmup_steps=20,
        learning_rate=1e-4, lr_scheduler_type='cosine',
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, save_steps=200, save_total_limit=2,
        seed=42, report_to='none'))

print(f'Training 3B on {len(ds_3b)} examples...')
stats_3b = trainer_3b.train()
print(f'3B done — loss: {stats_3b.training_loss:.3f}, {stats_3b.metrics["train_runtime"]:.0f}s')

# Save 3B intermediate
INT_SAVE = os.path.join(WORK, 'intermediate-3b', 'final')
model_3b.save_pretrained(INT_SAVE)
tok_3b.save_pretrained(INT_SAVE)
print(f'3B intermediate saved to {INT_SAVE}')

In [ ]:
# --- Step 3b: Generate 3B teacher logits for the 0.5B student ---
# Now the trained 3B generates responses + logits for the 0.5B to learn from.

print('=== Progressive Step 2: Generating 3B logits for 0.5B ===')

FastLanguageModel.for_inference(model_3b)

teacher_3b_data = []
t0 = time.time()

for i, ex in enumerate(distilled):
    text = format_chatml(ex)['text']
    # Split at assistant marker to get prompt
    parts = text.split('<|im_start|>assistant\n')
    prompt_text = parts[0] + '<|im_start|>assistant\n'

    inputs = tok_3b(prompt_text, return_tensors='pt', truncation=True, max_length=1536).to(model_3b.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model_3b.generate(
            **inputs, max_new_tokens=768, temperature=0.3, top_p=0.9,
            do_sample=True, output_scores=True, return_dict_in_generate=True,
            pad_token_id=tok_3b.pad_token_id or tok_3b.eos_token_id)

    gen_ids = outputs.sequences[0][input_len:]
    response_3b = tok_3b.decode(gen_ids, skip_special_tokens=True).strip()

    # Save top-k logits
    logits_3b = []
    for score in outputs.scores:
        top_vals, top_ids = torch.topk(score[0].float(), TOP_K_LOGITS)
        logits_3b.append({'ids': top_ids.cpu(), 'vals': top_vals.cpu()})

    teacher_3b_data.append({
        'instruction': ex['instruction'],
        'response_3b': response_3b,
        'output_original': ex['output'],
        'token_ids': gen_ids.cpu(),
        'logits': logits_3b,
        'task_type': ex.get('task_type', 'detect'),
    })

    if (i + 1) % 100 == 0:
        rate = (i + 1) / (time.time() - t0) * 60
        print(f'  [{i+1}/{len(distilled)}] {rate:.0f}/min')

LOGIT_3B_PATH = os.path.join(WORK, 'teacher_3b_logits.pt')
torch.save(teacher_3b_data, LOGIT_3B_PATH)
print(f'Saved {len(teacher_3b_data)} records with logits')

# Unload 3B
del model_3b, tok_3b, trainer_3b
gc.collect(); torch.cuda.empty_cache()
print('3B unloaded.')

In [ ]:
# --- Step 3c: Train 0.5B with SFT + logit KD from 3B ---

print('=== Progressive Step 3: 3B → 0.5B (SFT + Logit KD) ===')

from unsloth import FastLanguageModel
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer
import torch.nn.functional as F

# Load student
model_05, tok_05 = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)

model_05 = FastLanguageModel.get_peft_model(
    model_05, r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

FastLanguageModel.for_training(model_05)

# Build dataset: use 3B teacher's best response as target text
teacher_3b_data = torch.load(LOGIT_3B_PATH)

sft_texts = []
for d in teacher_3b_data:
    # Use original best response (from big teachers), not 3B's response
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{d["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{d["output_original"]}<|im_end|>'
    )
    sft_texts.append({'text': text, 'task_type': d.get('task_type', 'detect')})

ds_05 = Dataset.from_list(sft_texts)

# --- Custom KD trainer ---
# We train with combined loss:
#   L = alpha * CE_loss(student, target_tokens) + (1-alpha) * T^2 * KL(teacher || student)
# But since 3B logits are top-k sparse, we approximate KL on the top-k only.

KD_ALPHA = 0.5   # balance: 0.5 text + 0.5 logit
KD_TEMP = 3.0    # softmax temperature for KD

# First: standard SFT pass (fast, establishes baseline)
from trl import SFTTrainer

trainer_05 = SFTTrainer(
    model=model_05, tokenizer=tok_05, train_dataset=ds_05,
    dataset_text_field='text', max_seq_length=2048, packing=True,
    args=TrainingArguments(
        output_dir=os.path.join(WORK, 'student-05b-sft'),
        num_train_epochs=3, per_device_train_batch_size=4,
        gradient_accumulation_steps=4, warmup_steps=30,
        learning_rate=2e-4, lr_scheduler_type='cosine',
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, save_steps=200, save_total_limit=2,
        seed=42, report_to='none'))

print(f'SFT pass on {len(ds_05)} examples, 3 epochs...')
stats_sft = trainer_05.train()
print(f'SFT done — loss: {stats_sft.training_loss:.3f}')

# --- Logit KD pass ---
# After SFT, do a focused KD pass using the 3B's logits.
# This is a manual training loop since SFTTrainer doesn't support KD loss.

print('\nLogit KD refinement pass...')
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = AdamW(
    [p for p in model_05.parameters() if p.requires_grad],
    lr=5e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=len(teacher_3b_data))

model_05.train()
kd_losses = []
vocab_size = tok_05.vocab_size

for i, record in enumerate(teacher_3b_data):
    if not record['logits']:
        continue

    # Build input: full prompt + target
    text = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{record["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{record["output_original"]}<|im_end|>'
    )
    inputs = tok_05(text, return_tensors='pt', truncation=True, max_length=2048).to(model_05.device)

    # Forward pass
    outputs = model_05(**inputs, labels=inputs['input_ids'])
    ce_loss = outputs.loss

    # Compute sparse KD loss on the output tokens
    # Student logits for generated positions
    student_logits = outputs.logits  # (1, seq_len, vocab)

    # Build sparse teacher distribution from top-k
    n_teacher_tokens = min(len(record['logits']), student_logits.shape[1] - 1)
    if n_teacher_tokens > 0:
        kd_loss = torch.tensor(0.0, device=model_05.device)
        count = 0
        # Compare on last n_teacher_tokens positions
        offset = student_logits.shape[1] - n_teacher_tokens
        for t in range(min(n_teacher_tokens, 128)):  # cap per-example
            pos = offset + t
            if pos >= student_logits.shape[1]:
                break

            tl = record['logits'][t]
            t_ids = tl['ids'].to(model_05.device)
            t_vals = tl['vals'].float().to(model_05.device)

            # Teacher soft distribution (top-k only)
            teacher_probs = F.softmax(t_vals / KD_TEMP, dim=-1)

            # Student log-probs at teacher's top-k positions
            s_logits_at_k = student_logits[0, pos, t_ids] / KD_TEMP
            student_log_probs = F.log_softmax(s_logits_at_k, dim=-1)

            kd_loss += F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
            count += 1

        if count > 0:
            kd_loss = kd_loss / count * (KD_TEMP ** 2)
            total_loss = KD_ALPHA * ce_loss + (1 - KD_ALPHA) * kd_loss
        else:
            total_loss = ce_loss
    else:
        total_loss = ce_loss

    total_loss.backward()

    if (i + 1) % 4 == 0:  # grad accumulation
        torch.nn.utils.clip_grad_norm_(model_05.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    kd_losses.append(total_loss.item())
    if (i + 1) % 200 == 0:
        avg = sum(kd_losses[-200:]) / len(kd_losses[-200:])
        print(f'  [{i+1}/{len(teacher_3b_data)}] KD loss: {avg:.4f}')

avg_kd = sum(kd_losses) / len(kd_losses) if kd_losses else 0
print(f'Logit KD done — avg loss: {avg_kd:.4f}')

# Save base student
BASE_SAVE = os.path.join(WORK, 'student-05b-base')
model_05.save_pretrained(BASE_SAVE)
tok_05.save_pretrained(BASE_SAVE)
print(f'Base student saved to {BASE_SAVE}')

In [ ]:
# ============================================================
# PHASE 4: Mixture of LoRA Experts (MoLoRA)
# ============================================================
# Train 4 task-specific LoRA adapters on the base student.
# At inference, a lightweight router picks the right expert.
#
# Experts: detect, exploit, harden, taint
# Router: keyword-based classifier (no extra params, zero VRAM)

from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

EXPERT_TASKS = ['detect', 'exploit', 'harden', 'taint']

# Split distilled data by task type
with open(DISTILLED_PATH) as f:
    all_distilled = [json.loads(l) for l in f if l.strip()]

task_datasets = {}
for task in EXPERT_TASKS:
    examples = [ex for ex in all_distilled if ex.get('task_type') == task]
    if not examples:
        print(f'WARNING: no examples for {task}, using all data')
        examples = all_distilled
    task_datasets[task] = Dataset.from_list(examples).map(format_chatml)
    print(f'  {task}: {len(task_datasets[task])} examples')

# Train each expert
for task in EXPERT_TASKS:
    print(f'\n{"="*60}\n  Training expert: {task}\n{"="*60}')

    # Fresh student for each expert (load base weights)
    expert_model, expert_tok = FastLanguageModel.from_pretrained(
        model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)

    expert_model = FastLanguageModel.get_peft_model(
        expert_model, r=16, lora_alpha=32, lora_dropout=0.05,  # smaller rank per expert
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none', use_gradient_checkpointing='unsloth', random_state=42)

    # Merge base adapter first, then add expert adapter on top
    # Load base weights into the model before adding expert LoRA
    base_adapter_path = os.path.join(BASE_SAVE, 'adapter_model.safetensors')
    if os.path.exists(base_adapter_path):
        from safetensors.torch import load_file
        base_weights = load_file(base_adapter_path)
        missing = expert_model.load_state_dict(base_weights, strict=False)

    FastLanguageModel.for_training(expert_model)

    expert_trainer = SFTTrainer(
        model=expert_model, tokenizer=expert_tok,
        train_dataset=task_datasets[task],
        dataset_text_field='text', max_seq_length=2048, packing=True,
        args=TrainingArguments(
            output_dir=os.path.join(WORK, f'expert-{task}'),
            num_train_epochs=4,  # more epochs on smaller task-specific data
            per_device_train_batch_size=4,
            gradient_accumulation_steps=2,
            warmup_steps=10,
            learning_rate=3e-4,  # higher LR for focused expert
            lr_scheduler_type='cosine',
            weight_decay=0.01,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            logging_steps=25, save_total_limit=1,
            seed=42, report_to='none'))

    stats = expert_trainer.train()
    print(f'  {task} expert — loss: {stats.training_loss:.3f}')

    expert_save = os.path.join(WORK, f'expert-{task}', 'final')
    expert_model.save_pretrained(expert_save)
    expert_tok.save_pretrained(expert_save)

    del expert_model, expert_tok, expert_trainer
    gc.collect(); torch.cuda.empty_cache()

print(f'\nAll {len(EXPERT_TASKS)} experts trained!')

# --- Build router ---
# Zero-parameter keyword router — no extra VRAM

ROUTER_CODE = '''
TASK_KEYWORDS = {
    'detect': ['identify', 'vulnerability', 'find', 'detect', 'what is', 'CWE', 'scan',
               'what\'s wrong', 'security issue', 'flaw', 'weakness', 'bug', 'review', 'analyze'],
    'exploit': ['exploit', 'proof of concept', 'PoC', 'attack', 'payload', 'bypass',
                'demonstrate', 'craft', 'weaponize', 'inject', 'abuse'],
    'harden': ['fix', 'harden', 'secure', 'remediat', 'mitigat', 'patch', 'rewrite',
               'safe version', 'prevent', 'protect', 'sanitize', 'validate'],
    'taint': ['trace', 'taint', 'flow', 'data flow', 'source', 'sink', 'propagat',
              'reaches', 'user input', 'untrusted', 'control flow'],
}

def route(prompt: str) -> str:
    pl = prompt.lower()
    scores = {task: sum(1 for kw in kws if kw.lower() in pl)
              for task, kws in TASK_KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'detect'
'''

router_path = os.path.join(WORK, 'router.py')
with open(router_path, 'w') as f:
    f.write(ROUTER_CODE)
print(f'Router saved to {router_path}')

In [ ]:
# ============================================================
# PHASE 5: DPO Refinement
# ============================================================
# Generate preference pairs:
#   chosen  = best teacher response (from Phase 2)
#   rejected = raw 0.5B student response (before DPO)
# This aligns the student's generation style to teacher quality.

from unsloth import FastLanguageModel
from trl import DPOTrainer, DPOConfig
from datasets import Dataset

print('=== Phase 5: DPO Refinement ===')

# Load base student (post SFT+KD, pre-expert)
model_dpo, tok_dpo = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)

model_dpo = FastLanguageModel.get_peft_model(
    model_dpo, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', use_gradient_checkpointing='unsloth', random_state=42)

# Load base SFT weights
base_adapter_path = os.path.join(BASE_SAVE, 'adapter_model.safetensors')
if os.path.exists(base_adapter_path):
    from safetensors.torch import load_file
    model_dpo.load_state_dict(load_file(base_adapter_path), strict=False)

# Generate rejected responses from the current student
print('Generating student responses for DPO pairs...')
FastLanguageModel.for_inference(model_dpo)

with open(DISTILLED_PATH) as f:
    distilled = [json.loads(l) for l in f if l.strip()]

# Use a subset for DPO (it's expensive)
dpo_subset = distilled[:min(1000, len(distilled))]
dpo_pairs = []

for i, ex in enumerate(dpo_subset):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{ex["instruction"]}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tok_dpo(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model_dpo.device)

    with torch.no_grad():
        outputs = model_dpo.generate(
            **inputs, max_new_tokens=512, temperature=0.7,
            do_sample=True, top_p=0.9,
            pad_token_id=tok_dpo.pad_token_id or tok_dpo.eos_token_id)

    rejected = tok_dpo.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    dpo_pairs.append({
        'prompt': prompt,
        'chosen': ex['output'],
        'rejected': rejected,
    })

    if (i + 1) % 100 == 0:
        print(f'  [{i+1}/{len(dpo_subset)}] pairs generated')

dpo_dataset = Dataset.from_list(dpo_pairs)
print(f'DPO dataset: {len(dpo_dataset)} preference pairs')

# DPO training
FastLanguageModel.for_training(model_dpo)

dpo_trainer = DPOTrainer(
    model=model_dpo,
    ref_model=None,  # use implicit reference (LoRA base)
    train_dataset=dpo_dataset,
    tokenizer=tok_dpo,
    args=DPOConfig(
        output_dir=os.path.join(WORK, 'student-dpo'),
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-5,  # conservative LR for DPO
        lr_scheduler_type='cosine',
        warmup_steps=10,
        beta=0.1,
        max_length=2048,
        max_prompt_length=1024,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        seed=42,
        report_to='none',
    ),
)

print('DPO training...')
dpo_stats = dpo_trainer.train()
print(f'DPO done — loss: {dpo_stats.training_loss:.3f}')

# Save DPO-refined adapter
DPO_SAVE = os.path.join(WORK, 'student-dpo', 'final')
model_dpo.save_pretrained(DPO_SAVE)
tok_dpo.save_pretrained(DPO_SAVE)

del model_dpo, tok_dpo, dpo_trainer
gc.collect(); torch.cuda.empty_cache()
print(f'DPO adapter saved to {DPO_SAVE}')

In [ ]:
# ============================================================
# PHASE 6: Benchmark — compare everything
# ============================================================

from unsloth import FastLanguageModel
import time, gc, torch

BENCH = [
    {'id': 'sqli', 'task': 'Identify the vulnerability and CWE:\nquery = "SELECT * FROM users WHERE name = \'%s\'" % request.args.get(\'name\')\ncursor.execute(query)',
     'expect': ['SQL injection', 'CWE-89', 'parameterized', 'prepared statement']},
    {'id': 'xss', 'task': 'Write a PoC exploit for this reflected XSS:\n@app.route(\'/search\')\ndef search():\n    q = request.args.get(\'q\', \'\')\n    return f\'<h1>Results for {q}</h1>\'',
     'expect': ['<script>', 'alert', 'document.cookie', 'XSS', 'CWE-79']},
    {'id': 'traversal', 'task': 'What vulnerability, CWE, and fix?\n@app.route(\'/download\')\ndef download():\n    filename = request.args.get(\'file\')\n    return send_file(f\'/uploads/{filename}\')',
     'expect': ['path traversal', 'CWE-22', 'directory traversal', 'os.path.basename', 'sanitize']},
    {'id': 'harden', 'task': 'Write a hardened version:\ndef login():\n    user = request.form[\'username\']\n    pw = request.form[\'password\']\n    cur.execute("SELECT * FROM users WHERE user=\'%s\' AND pass=\'%s\'" % (user, pw))\n    return \'OK\' if cur.fetchone() else \'Fail\'',
     'expect': ['parameterized', '?', '%s', 'execute(', 'bcrypt', 'hash']},
    {'id': 'taint', 'task': 'Trace the taint flow — where does user input reach a dangerous sink?\nname = request.cookies.get(\'username\')\ntemplate = \'<div>Hello, \' + name + \'</div>\'\nreturn render_template_string(template)',
     'expect': ['SSTI', 'template injection', 'cookie', 'render_template_string', 'CWE-94', 'Jinja']},
]

def bench_model(model, tokenizer, label):
    FastLanguageModel.for_inference(model)
    results = []
    for p in BENCH:
        gc.collect(); torch.cuda.empty_cache()
        prompt = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{p["task"]}<|im_end|>\n<|im_start|>assistant\n'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=768, temperature=0.1, top_p=0.9,
                                 do_sample=True, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        rl = resp.lower()
        hits = [k for k in p['expect'] if k.lower() in rl]
        results.append({'id': p['id'], 'score': len(hits)/len(p['expect']), 'hits': hits})
    avg = sum(r['score'] for r in results) / len(results)
    print(f'\n  {label}: {avg:.0%}')
    for r in results:
        print(f'    {r["id"]:<12s} {r["score"]:>5.0%}  hits: {r["hits"]}')
    return avg

print('='*60)
print('  BENCHMARK COMPARISON')
print('='*60)

# Benchmark raw 0.5B
m_raw, t_raw = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)
raw_avg = bench_model(m_raw, t_raw, '0.5B raw (no training)')
del m_raw, t_raw; gc.collect(); torch.cuda.empty_cache()

# Benchmark base SFT+KD
m_sft, t_sft = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)
m_sft = FastLanguageModel.get_peft_model(
    m_sft, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', random_state=42)
from safetensors.torch import load_file
m_sft.load_state_dict(load_file(os.path.join(BASE_SAVE, 'adapter_model.safetensors')), strict=False)
sft_avg = bench_model(m_sft, t_sft, '0.5B + SFT + logit KD')
del m_sft, t_sft; gc.collect(); torch.cuda.empty_cache()

# Benchmark DPO
m_dpo, t_dpo = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)
m_dpo = FastLanguageModel.get_peft_model(
    m_dpo, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', random_state=42)
m_dpo.load_state_dict(load_file(os.path.join(DPO_SAVE, 'adapter_model.safetensors')), strict=False)
dpo_avg = bench_model(m_dpo, t_dpo, '0.5B + SFT + KD + DPO')
del m_dpo, t_dpo; gc.collect(); torch.cuda.empty_cache()

# Summary
print(f'\n{"="*60}')
print(f'  SUMMARY')
print(f'{"="*60}')
print(f'  0.5B raw:            {raw_avg:>5.0%}')
print(f'  0.5B + SFT + KD:     {sft_avg:>5.0%}')
print(f'  0.5B + SFT + KD + DPO: {dpo_avg:>5.0%}')
print(f'  (MoLoRA experts add task-specific gains on top)')

In [ ]:
# ============================================================
# PHASE 6b: Export — GGUF + Ollama Modelfile + MoLoRA bundle
# ============================================================

from unsloth import FastLanguageModel
import shutil

# Export the DPO-refined base adapter as GGUF
print('Exporting GGUF...')
m_exp, t_exp = FastLanguageModel.from_pretrained(
    model_name=STUDENT_ID, max_seq_length=2048, load_in_4bit=True, dtype=None)
m_exp = FastLanguageModel.get_peft_model(
    m_exp, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', random_state=42)
m_exp.load_state_dict(load_file(os.path.join(DPO_SAVE, 'adapter_model.safetensors')), strict=False)

m_exp.save_pretrained_merged('/content/owen-distilled-merged', t_exp, save_method='merged_16bit')
m_exp.save_pretrained_gguf('/content/owen-distilled-gguf', t_exp, quantization_method='q4_k_m')

import glob
for f in glob.glob('/content/owen-distilled-gguf/*.gguf'):
    size = os.path.getsize(f) / 1024**3
    print(f'  {os.path.basename(f)}: {size:.2f} GB')

# Modelfile for Ollama
MODELFILE = '''FROM ./owen-coder-distilled.gguf
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 2048
SYSTEM """You are Owen Coder, a security-focused code analysis model distilled from a council of 5 expert models (27B, 32B, 16B, 7B, 3.8B). You detect vulnerabilities, write exploit PoCs, perform taint analysis, and generate hardening recommendations. Be precise, technical, and exhaustive. Reference exact line numbers and CWE IDs."""
'''
with open('/content/owen-distilled-gguf/Modelfile', 'w') as f:
    f.write(MODELFILE)

# Bundle MoLoRA experts for download
expert_bundle = os.path.join(WORK, 'molora-experts')
os.makedirs(expert_bundle, exist_ok=True)
for task in EXPERT_TASKS:
    src = os.path.join(WORK, f'expert-{task}', 'final')
    dst = os.path.join(expert_bundle, task)
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
shutil.copy2(router_path, expert_bundle)
shutil.make_archive('/content/molora-experts', 'zip', expert_bundle)

# Upload to HF Hub
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get('HF_TOKEN')
api = HfApi(token=token)
username = api.whoami()['name']

repo_id = f'{username}/owen-coder-distilled-v2'
api.create_repo(repo_id, exist_ok=True, private=True)
api.upload_folder(folder_path=DPO_SAVE, repo_id=repo_id, repo_type='model')
print(f'\nUploaded to https://huggingface.co/{repo_id}')

del m_exp, t_exp; gc.collect(); torch.cuda.empty_cache()

print(f'\n{"="*60}')
print(f'  DONE! Files ready for download:')
print(f'{"="*60}')
print(f'  GGUF:    /content/owen-distilled-gguf/')
print(f'  Experts: /content/molora-experts.zip')
print(f'  Hub:     https://huggingface.co/{repo_id}')
print(f'\n  To use with Ollama:')
print(f'    cd /content/owen-distilled-gguf')
print(f'    ollama create owen-coder-distilled -f Modelfile')
print(f'    ollama run owen-coder-distilled')

In [ ]:
# Download everything
from google.colab import files
import glob

for f in glob.glob('/content/owen-distilled-gguf/*.gguf'):
    print(f'Downloading {os.path.basename(f)}...')
    files.download(f)

files.download('/content/owen-distilled-gguf/Modelfile')
files.download('/content/molora-experts.zip')

print('All done!')